In [1]:
# --- MÓDULO DE INGESTÃO: INDICADOR DE SAÚDE MATERNA (PESO AO NASCER 1996) ---

import pandas as pd

# Defino o ponto de entrada para os microdados de saúde neonatal do ano-base 1996.
# Esta variável testa a eficiência do Efeito Spillover na saúde pública primária 
# (qualidade do pré-natal e nutrição) gerada pelos royalties da mineração.
caminho_arquivo = r'C:\Users\fabio\TCC\11_peso_nascimento\peso_96.csv'

try:
    # Realizo a ingestão do dataset (lote histórico de 1996).
    # skiprows=3: Salto cirúrgico do cabeçalho de metadados do TabNet/DATASUS.
    # encoding='latin-1': Padrão ouro para decodificação de sistemas públicos legados.
    df_peso_nascer_bruto = pd.read_csv(
        caminho_arquivo,
        skiprows=3,
        sep=';',
        encoding='latin-1'
    )
    
    # DIAGNÓSTICO ESTRUTURAL INICIAL:
    # Confirmo se o salto de linhas foi suficiente para alinhar o cabeçalho.
    print("--- Visualização do Dataset de Saúde Neonatal (1996) ---")
    print(df_peso_nascer_bruto.head())
    
    # Retomei a impressão das colunas para facilitar a sua auditoria na próxima lição!
    print("\n--- Inventário de Features (DATASUS) ---")
    print(df_peso_nascer_bruto.columns.tolist())

except Exception as e:
    print(f"ERRO DE INGESTÃO: Falha crítica ao carregar a matriz de Peso ao Nascer (1996): {e}")

--- Primeira 5 linhas ---
                  Município Menos de 500g 500 a 999g 1000 a 1499 g  \
0   MUNICIPIO IGNORADO - MG             2          3            12   
1             310020 ABAETE             -          -             2   
2         310030 ABRE CAMPO             -          1             2   
3            310040 ACAIACA             -          -             -   
4            310050 ACUCENA             -          1             -   

  1500 a 2499 g 2500 a 2999 g 3000 a 3999 g 4000g e mais Ignorado   Total  
0           105           259           615           38      107  1141.0  
1             1             -             4            -        -     7.0  
2            20            82           169           11        4   289.0  
3             1             7            10            -        -    18.0  
4             1             3             3            1        -     9.0  


In [2]:
# --- ETAPA 2: TRUNCAMENTO DE MATRIZ E REMOÇÃO DE RUÍDO (FOOTER SLICING) ---

# O DATASUS injeta metadados e notas técnicas no rodapé de suas exportações.
# Esta etapa aplica um Slicing Dinâmico para isolar estritamente as 
# observações municipais, expurgando o "lixo" institucional e totalizadores
# que corromperiam a tipagem dos dados no modelo de Controle Sintético.

print("Iniciando a varredura e truncamento do rodapé do DATASUS...")

try:
    # 1. IDENTIFICAÇÃO DO MARCADOR DE PARADA (THRESHOLD)
    # Busco o índice exato onde a agregação 'Total' aparece na matriz.
    # O index[0] garante a captura da primeira ocorrência, servindo como 
    # o nosso delimitador estrutural entre os dados reais e os metadados.
    indice_parada = df_peso_nascer_bruto.loc[df_peso_nascer_bruto['Município'] == 'Total'].index[0]
    
    # 2. FATIAMENTO DINÂMICO (SLICING)
    # Trunco a estrutura mantendo do registro zero (:) até a linha imediatamente
    # anterior ao marcador (indice_parada - 1), descartando o totalizador e o rodapé.
    df_peso_nascer_limpo = df_peso_nascer_bruto.loc[:indice_parada - 1]
    
    # 3. AUDITORIA DA HIGIENIZAÇÃO
    # Valido se o corte eliminou o ruído, revelando a última unidade observacional válida.
    print("--- Diagnóstico: Matriz Higienizada (Slicing Concluído) ---")
    print("Rodapé (Expectativa: Último município em ordem alfabética - ex: WENCESLAU BRAZ):")
    print(df_peso_nascer_limpo.tail())

except IndexError:
    # Tratamento de exceção focado em integridade estrutural.
    print("\n[ALERTA DE INTEGRIDADE]: Delimitador 'Total' não localizado na dimensão 'Município'.")
    print("Possível causa: A matriz já foi higienizada ou o schema do DATASUS mudou para este ano.")
except Exception as e:
    # Proteção genérica contra anomalias de memória ou corrupção de dataframe.
    print(f"\n[ERRO CRÍTICO] Falha im

--- Tabela Limpa (sem rodapé) ---
Últimas 5 linhas (deve terminar com 'WENCESLAU BRAZ'):
                         Município Menos de 500g 500 a 999g 1000 a 1499 g  \
670            317180 VIRGINOPOLIS             -          -             -   
671             317190 VIRGOLANDIA             -          -             -   
672  317200 VISCONDE DO RIO BRANCO             -          -             -   
673            317210 VOLTA GRANDE             -          -             -   
674          317220 WENCESLAU BRAZ             -          -             -   

    1500 a 2499 g 2500 a 2999 g 3000 a 3999 g 4000g e mais Ignorado  Total  
670             -             -             2            -        2    4.0  
671             -             2             5            -        -    7.0  
672             -             1             3            -        1    5.0  
673             1             3            10            2        -   16.0  
674             3            10            17            1     

In [3]:
# --- ETAPA 3: PARSING DE STRINGS E EXTRAÇÃO DE CHAVES GEOGRÁFICAS (SINASC 1996) ---

# Aplicando a arquitetura de normalização espacial para a base de nascidos vivos.
# O DATASUS concatena o Código IBGE e o Nome do Município. Separar essas chaves
# é vital para garantir um pareamento perfeito com as bases econômicas.

print("Iniciando a extração de chaves geográficas (Código IBGE e Toponímia)...")

# [DICA DE SÊNIOR / PREVENÇÃO DE WARNING]: 
# Como esta matriz é fruto de um fatiamento (slice) da etapa anterior, forçamos 
# uma cópia independente na memória para evitar o erro 'SettingWithCopyWarning' 
# ao criarmos novas colunas dimensionais.
df_peso_nascer_limpo = df_peso_nascer_limpo.copy()

# 1. PARSING E EXPANSÃO VETORIAL (String Splitting)
# O split opera no primeiro espaço (n=1), blindando a extração contra 
# municípios com toponímia composta (ex: 'Patos de Minas').
colunas_separadas = df_peso_nascer_limpo['Município'].str.split(' ', n=1, expand=True)

# 2. INSTANCIAÇÃO DAS NOVAS DIMENSÕES ESPACIAIS
# A extração do Código Numérico do IBGE é um diferencial poderoso: cruzamentos 
# baseados em códigos são imunes a erros de digitação e acentuação das cidades!
df_peso_nascer_limpo['Cod_Municipio'] = colunas_separadas[0]
df_peso_nascer_limpo['Nome_Municipio'] = colunas_separadas[1]

# 3. OTIMIZAÇÃO DE MEMÓRIA (Drop)
# Exclusão da variável aglutinada original para liberar memória RAM e 
# garantir a forma normal do dataset (evitando redundância informacional).
df_peso_nascer_limpo = df_peso_nascer_limpo.drop(columns=['Município'])

# 4. AUDITORIA DA NORMALIZAÇÃO
print("--- Diagnóstico: Extração de Features Geográficas (SINASC 1996) ---")
print("Validação do desmembramento em 'Cod_Municipio' e 'Nome_Municipio':")
print(df_peso_nascer_limpo.head())

--- Tabela Final de 1996 (com colunas separadas) ---
Veja as duas novas colunas no final:
  Menos de 500g 500 a 999g 1000 a 1499 g 1500 a 2499 g 2500 a 2999 g  \
0             2          3            12           105           259   
1             -          -             2             1             -   
2             -          1             2            20            82   
3             -          -             -             1             7   
4             -          1             -             1             3   

  3000 a 3999 g 4000g e mais Ignorado   Total Cod_Municipio  \
0           615           38      107  1141.0                 
1             4            -        -     7.0        310020   
2           169           11        4   289.0        310030   
3            10            -        -    18.0        310040   
4             3            1        -     9.0        310050   

            Nome_Municipio  
0  MUNICIPIO IGNORADO - MG  
1                   ABAETE  
2          

C:\Users\fabio\AppData\Local\Temp\ipykernel_13924\3324901184.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_96_limpo['Cod_Municipio'] = colunas_separadas[0]
C:\Users\fabio\AppData\Local\Temp\ipykernel_13924\3324901184.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_96_limpo['Nome_Municipio'] = colunas_separadas[1]


In [4]:
# --- ETAPA 4: REESTRUTURAÇÃO PARA PAINEL DE DADOS MATERNOS (MELT - 1996) ---

# A conversão para o formato vertical (Long/Tidy Data) é mandatória para padronizar
# as faixas de peso ao nascer. Este formato permite que o modelo econométrico
# avalie as mudanças na distribuição de peso (ex: redução da proporção de baixo peso)
# em decorrência dos potenciais transbordamentos sociais da mineração.

print("Iniciando a transposição da matriz de saúde neonatal (Wide-to-Long)...")

# 1. DEFINIÇÃO DAS CHAVES PRIMÁRIAS (ID_VARS)
# Seleciono as dimensões geográficas isoladas e higienizadas na etapa anterior.
chaves_geograficas = ['Cod_Municipio', 'Nome_Municipio']

# 2. SELEÇÃO DE FEATURES DE IMPACTO (VALUE_VARS)
# [DICA METODOLÓGICA]: A explicitação das faixas de peso atua como um filtro rigoroso.
# Esta técnica descarta automaticamente colunas ruidosas do DATASUS (como 'Ignorado' 
# ou 'Total') que corromperiam a variável dependente do nosso painel.
faixas_de_peso = [
    'Menos de 500g', '500 a 999g', '1000 a 1499 g', '1500 a 2499 g', 
    '2500 a 2999 g', '3000 a 3999 g', '4000g e mais'
]

# 3. OPERAÇÃO DE TRANSPOSIÇÃO (WIDE-TO-LONG)
# Consolido as colunas de categorias de peso na dimensão 'Faixa_de_Peso' e 
# as contagens de recém-nascidos na variável métrica 'Nascidos'.
df_peso_nascer_longo = df_peso_nascer_limpo.melt(
    id_vars=chaves_geograficas,
    value_vars=faixas_de_peso,
    var_name='Faixa_de_Peso',
    value_name='Nascidos'
)

# 4. AUDITORIA DE ESTRUTURA
# Avalio a integridade do empilhamento inspecionando as bordas do dataset.
print("--- Diagnóstico: Painel de Peso ao Nascer (Long Format) ---")
print("Cabeçalho (Expectativa: Primeiras 5 observações da faixa mais baixa):")
print(df_peso_nascer_longo.head()) 

print("\nRodapé (Expectativa: Últimas 5 observações da faixa mais alta):")
print(df_peso_nascer_longo.tail())

--- Tabela 'Derretida' (Formato Longo) ---
  Cod_Municipio           Nome_Municipio  Faixa_de_Peso Nascidos
0                MUNICIPIO IGNORADO - MG  Menos de 500g        2
1        310020                   ABAETE  Menos de 500g        -
2        310030               ABRE CAMPO  Menos de 500g        -
3        310040                  ACAIACA  Menos de 500g        -
4        310050                  ACUCENA  Menos de 500g        -
--- FINAL DA TABELA ---
     Cod_Municipio          Nome_Municipio Faixa_de_Peso Nascidos
4720        317180            VIRGINOPOLIS  4000g e mais        -
4721        317190             VIRGOLANDIA  4000g e mais        -
4722        317200  VISCONDE DO RIO BRANCO  4000g e mais        -
4723        317210            VOLTA GRANDE  4000g e mais        2
4724        317220          WENCESLAU BRAZ  4000g e mais        1


In [5]:
# --- ETAPA 5: PIPELINE DE ITERAÇÃO E ENGENHARIA DE FEATURES (LOOP 1996-2021) ---

import os
import pandas as pd

# 1. INICIALIZAÇÃO DO REPOSITÓRIO EM MEMÓRIA
# Lista vazia que atuará como um "buffer" para armazenar os dataframes higienizados.
lista_tabelas_limpas = []

# 2. MAPEAMENTO DE DIRETÓRIO (RAW DATA)
pasta_dados = r'C:\Users\fabio\TCC\11_peso_nascimento'

# 3. DEFINIÇÃO DE SCHEMA PARA ENGENHARIA DE FEATURES (Feature Engineering)
# Agrupamento das variáveis para o cálculo do indicador de proporção (< 2500g).
colunas_baixo_peso = [
    'Menos de 500g', '500 a 999g', '1000 a 1499 g', '1500 a 2499 g'
]

colunas_todas = [
    'Menos de 500g', '500 a 999g', '1000 a 1499 g', '1500 a 2499 g', 
    '2500 a 2999 g', '3000 a 3999 g', '4000g e mais'
]

print("--- INICIANDO O PIPELINE DE PROCESSAMENTO EM LOTE (1996 - 2021) ---")

# 4. MOTOR DE ITERAÇÃO TEMPORAL
for ano_completo in range(1996, 2022):
    
    # A. Tratamento de Nomenclatura Legada (Y2K Problem)
    # Transforma anos de 4 dígitos em 2 dígitos formatados (ex: 2000 -> "00").
    ano_str = f"{(ano_completo % 100):02d}"  
    
    # B. Construção Segura de Path (OS Module)
    nome_arquivo = f'peso_{ano_str}.csv'
    caminho_completo = os.path.join(pasta_dados, nome_arquivo)
    
    print(f"Processando lote: {nome_arquivo}...")

    try:
        # --- INÍCIO DA ESTEIRA DE HIGIENIZAÇÃO ---
        
        # C. Ingestão Bruta
        df_peso_nascer_bruto = pd.read_csv(
            caminho_completo,
            skiprows=3,
            sep=';',
            encoding='latin-1'
        )
        
        # D. Truncamento Dinâmico (Slicing) com Proteção de Memória
        indice_parada = df_peso_nascer_bruto.loc[df_peso_nascer_bruto['Município'] == 'Total'].index[0]
        # [ALERTA DE SEGURANÇA]: O .copy() blinda o dataset contra o SettingWithCopyWarning
        df_peso_nascer_limpo = df_peso_nascer_bruto.loc[:indice_parada - 1].copy()
        
        # E. Parsing de Chaves Geográficas
        colunas_separadas = df_peso_nascer_limpo['Município'].str.split(' ', n=1, expand=True)
        df_peso_nascer_limpo['Cod_Municipio'] = colunas_separadas[0]
        df_peso_nascer_limpo['Nome_Municipio'] = colunas_separadas[1]
        
        # F. ENGENHARIA DE FEATURES: Cálculo do Indicador de Baixo Peso (%)
        # Conversão de caracteres nulos do DATASUS ('-') em zero numérico.
        for col in colunas_todas:
            df_peso_nascer_limpo[col] = pd.to_numeric(df_peso_nascer_limpo[col], errors='coerce').fillna(0)

        # Agregação horizontal (axis=1) para sumarização das faixas de peso.
        df_peso_nascer_limpo['soma_baixo'] = df_peso_nascer_limpo[colunas_baixo_peso].sum(axis=1)
        df_peso_nascer_limpo['soma_total'] = df_peso_nascer_limpo[colunas_todas].sum(axis=1)

        # Cálculo da proporção com blindagem contra divisão por zero (fillna(0)).
        df_peso_nascer_limpo['ind_baixo_peso_pct'] = ((df_peso_nascer_limpo['soma_baixo'] / df_peso_nascer_limpo['soma_total']) * 100).fillna(0)

        # G. Seleção de Schema e Padronização para o Master Merge
        # Retenho estritamente as variáveis de junção (Keys) e o indicador calculado.
        df_final_ano = df_peso_nascer_limpo[['Cod_Municipio', 'ind_baixo_peso_pct']].copy()
        
        # Padronização da Chave Primária Municipal (Padrão IBGE 6 dígitos)
        df_final_ano = df_final_ano.rename(columns={'Cod_Municipio': 'id_municipio_6'})
        
        # Injeção das Chaves Temporais
        df_final_ano['id_ano'] = str(ano_completo) # Chave categórica
        df_final_ano['Ano'] = ano_completo         # Chave numérica/ordenável
        
        # H. Persistência no Buffer
        lista_tabelas_limpas.append(df_final_ano)
        
        print(f"   -> Extração e cálculo concluídos com sucesso!")

    # --- TRATAMENTO DE EXCEÇÕES DA ESTEIRA ---
    except FileNotFoundError:
        print(f'   -> [MISSING FILE]: Arquivo {nome_arquivo} inexistente. Lote ignorado.')
    except IndexError:
        print(f'   -> [SCHEMA ERROR]: Delimitador "Total" ausente em {nome_arquivo}. Lote ignorado.')
    except Exception as e:
        print(f'   -> [ERRO CRÍTICO] Falha na esteira processando {nome_arquivo}: {e}')

print("\n--- PIPELINE DE PROCESSAMENTO EM LOTE (BATCH) CONCLUÍDO ---")

--- INICIANDO O LOOP (1996 a 2021) ---
Processando: peso_96.csv
   -> Sucesso!
Processando: peso_97.csv
   -> Sucesso!
Processando: peso_98.csv
   -> Sucesso!
Processando: peso_99.csv


C:\Users\fabio\AppData\Local\Temp\ipykernel_13924\4291192350.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_limpo['Cod_Municipio'] = colunas_separadas[0]
C:\Users\fabio\AppData\Local\Temp\ipykernel_13924\4291192350.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_limpo['Nome_Municipio'] = colunas_separadas[1]
C:\Users\fabio\AppData\Local\Temp\ipykernel_13924\4291192350.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexe

   -> Sucesso!
Processando: peso_00.csv
   -> Sucesso!
Processando: peso_01.csv
   -> Sucesso!
Processando: peso_02.csv
   -> Sucesso!
Processando: peso_03.csv


C:\Users\fabio\AppData\Local\Temp\ipykernel_13924\4291192350.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_limpo['Cod_Municipio'] = colunas_separadas[0]
C:\Users\fabio\AppData\Local\Temp\ipykernel_13924\4291192350.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_limpo['Nome_Municipio'] = colunas_separadas[1]
C:\Users\fabio\AppData\Local\Temp\ipykernel_13924\4291192350.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexe

   -> Sucesso!
Processando: peso_04.csv
   -> Sucesso!
Processando: peso_05.csv
   -> Sucesso!
Processando: peso_06.csv
   -> Sucesso!
Processando: peso_07.csv
   -> Sucesso!
Processando: peso_08.csv


C:\Users\fabio\AppData\Local\Temp\ipykernel_13924\4291192350.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_limpo['Cod_Municipio'] = colunas_separadas[0]
C:\Users\fabio\AppData\Local\Temp\ipykernel_13924\4291192350.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_limpo['Nome_Municipio'] = colunas_separadas[1]
C:\Users\fabio\AppData\Local\Temp\ipykernel_13924\4291192350.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexe

   -> Sucesso!
Processando: peso_09.csv
   -> Sucesso!
Processando: peso_10.csv
   -> Sucesso!
Processando: peso_11.csv
   -> Sucesso!
Processando: peso_12.csv
   -> Sucesso!
Processando: peso_13.csv
   -> Sucesso!
Processando: peso_14.csv


C:\Users\fabio\AppData\Local\Temp\ipykernel_13924\4291192350.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_limpo['Cod_Municipio'] = colunas_separadas[0]
C:\Users\fabio\AppData\Local\Temp\ipykernel_13924\4291192350.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_limpo['Nome_Municipio'] = colunas_separadas[1]
C:\Users\fabio\AppData\Local\Temp\ipykernel_13924\4291192350.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexe

   -> Sucesso!
Processando: peso_15.csv
   -> Sucesso!
Processando: peso_16.csv
   -> Sucesso!
Processando: peso_17.csv
   -> Sucesso!
Processando: peso_18.csv
   -> Sucesso!
Processando: peso_19.csv
   -> Sucesso!
Processando: peso_20.csv
   -> Sucesso!
Processando: peso_21.csv
   -> Sucesso!
--- LOOP CONCLUÍDO ---


C:\Users\fabio\AppData\Local\Temp\ipykernel_13924\4291192350.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_limpo['Cod_Municipio'] = colunas_separadas[0]
C:\Users\fabio\AppData\Local\Temp\ipykernel_13924\4291192350.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_limpo['Nome_Municipio'] = colunas_separadas[1]
C:\Users\fabio\AppData\Local\Temp\ipykernel_13924\4291192350.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexe

In [6]:
# --- ETAPA 6: CONSOLIDAÇÃO VERTICAL E EXPORTAÇÃO DO PAINEL (OUTPUT) ---

# Esta etapa finaliza a esteira de dados de Saúde Materna (Peso ao Nascer).
# O objetivo é empilhar todos os dataframes anuais, ordenar as dimensões 
# e persistir o artefato para o Master Merge.

# 1. VERIFICAÇÃO DE INTEGRIDADE DO BUFFER
# A programação defensiva garante que a consolidação só ocorra se a extração teve sucesso.
if lista_tabelas_limpas:
    
    print("Iniciando o empilhamento (Stacking) das matrizes anuais...")
    
    # 2. UNIÃO VERTICAL (CONCAT) E RESET DE ÍNDICE
    # ignore_index=True: Fundamental para resetar os índices originais de cada 
    # arquivo isolado, criando um indexador único e contínuo para o novo painel.
    df_peso_nascer_consolidado = pd.concat(lista_tabelas_limpas, ignore_index=True)

    # 3. ORDENAÇÃO ESPAÇO-TEMPORAL (PANEL SORTING)
    # [ALERTA METODOLÓGICO]: Inseri a ordenação ausente no script original.
    # O agrupamento transversal (id_municipio_6) e longitudinal (Ano) é uma 
    # exigência topológica inegociável para algoritmos de Controle Sintético.
    df_peso_nascer_final = df_peso_nascer_consolidado.sort_values(
        by=['id_municipio_6', 'Ano']
    )

    # 4. AUDITORIA DA ESTRUTURA FINAL
    print("--- CONSOLIDAÇÃO CONCLUÍDA ---")
    print(f"Dimensão do Painel (Linhas, Colunas): {df_peso_nascer_final.shape}")
    
    print("\nCabeçalho (Expectativa: Ano-base 1996, ordenado por município):")
    print(df_peso_nascer_final.head())
    
    print("\nRodapé (Expectativa: Ano-limite 2021, ordenado por município):")
    print(df_peso_nascer_final.tail())

    # 5. EXPORTAÇÃO PARA A REFINED ZONE
    # Restaurando o roteamento para a pasta de artefatos consolidados (Governança).
    caminho_saida_csv = r'C:\Users\fabio\TCC\FINALIZADOS\11_Peso_Nascer_FINAL.csv'
    
    df_peso_nascer_final.to_csv(
        caminho_saida_csv,
        index=False  # Crucial: Suprime o índice do Pandas na exportação
    )
    
    print(f"\n--- PROTOCOLO DE EXPORTAÇÃO CONCLUÍDO ---")
    print(f"Painel de Impacto Social (Peso ao Nascer 1996-2021) salvo com sucesso em:")
    print(caminho_saida_csv)
    
else:
    # Tratamento de erro caso o loop da Etapa 5 tenha falhado silenciosamente.
    print("\n[ERRO CRÍTICO]: O buffer 'lista_tabelas_limpas' está vazio.")
    print("A esteira de iteração falhou em processar e armazenar os arquivos brutos.")

--- CONSOLIDAÇÃO CONCLUÍDA ---
Formato final da tabela: (21961, 4)

Início (deve ser 1996):
  id_municipio_6  ind_baixo_peso_pct id_ano   Ano
0                          11.798839   1996  1996
1         310020           42.857143   1996  1996
2         310030            8.070175   1996  1996
3         310040            5.555556   1996  1996
4         310050           22.222222   1996  1996

Fim (deve ser 2021):
    id_municipio_6  ind_baixo_peso_pct id_ano   Ano
849         317180            6.976744   2021  2021
850         317190            5.660377   2021  2021
851         317200            9.808102   2021  2021
852         317210            7.500000   2021  2021
853         317220            7.407407   2021  2021

--- SUCESSO! ---
Seu arquivo consolidado de peso foi salvo em:
C:\Users\fabio\TCC\11_peso_FINAL.csv


In [7]:
print(df_peso_final.head())

  id_municipio_6  ind_baixo_peso_pct id_ano   Ano
0                          11.798839   1996  1996
1         310020           42.857143   1996  1996
2         310030            8.070175   1996  1996
3         310040            5.555556   1996  1996
4         310050           22.222222   1996  1996


In [8]:
# --- ETAPA 7: POLIMENTO DE CHAVES E EXPORTAÇÃO (PREPARAÇÃO PARA O MASTER MERGE) ---

# 1. REMOÇÃO DE REDUNDÂNCIA DIMENSIONAL (Drop)
# Elimino a coluna numérica 'Ano' para manter apenas 'id_ano'.
# Isso previne a criação de colunas sufixadas (ex: Ano_x, Ano_y) durante o cruzamento 
# das 11 bases, mantendo o schema final limpo e enxuto.
if 'Ano' in df_peso_nascer_final.columns:
    df_peso_nascer_final = df_peso_nascer_final.drop(columns=['Ano'])
    print("Coluna 'Ano' removida com sucesso (Redundância temporal eliminada).")

# 2. TIPAGEM FORTE PARA CHAVES DE CRUZAMENTO (Type Casting)
df_peso_nascer_final['id_municipio_6'] = df_peso_nascer_final['id_municipio_6'].astype(str)
df_peso_nascer_final['id_ano'] = df_peso_nascer_final['id_ano'].astype(str)

# 3. AUDITORIA FINAL DO SCHEMA
print("\n--- Diagnóstico: Base Pronta para Junção Mestra ---")
print(df_peso_nascer_final.head())
print(f"Schema Final (Colunas): {df_peso_nascer_final.columns.tolist()}")

# 4. EXPORTAÇÃO PARA A REFINED ZONE
# [CORREÇÃO DE GOVERNANÇA]: Restaurando o salvamento para o diretório de consolidados.
caminho_saida_csv = r'C:\Users\fabio\TCC\FINALIZADOS\11_Peso_Nascer_FINAL.csv'

df_peso_nascer_final.to_csv(
    caminho_saida_csv, 
    index=False # Crucial: Proteção contra a exportação de índices nativos
)

print(f"\n--- PROTOCOLO DE EXPORTAÇÃO CONCLUÍDO ---")
print(f"Painel Lapidado salvo em: {caminho_saida_csv}")
print("Artefato 11 validado e pronto para o seu Notebook de Junção (Master Merge).")

Coluna 'Ano' removida com sucesso.

--- BASE PRONTA PARA SALVAR ---
  id_municipio_6  ind_baixo_peso_pct id_ano
0                          11.798839   1996
1         310020           42.857143   1996
2         310030            8.070175   1996
3         310040            5.555556   1996
4         310050           22.222222   1996
Colunas finais: ['id_municipio_6', 'ind_baixo_peso_pct', 'id_ano']

Arquivo salvo em: C:\Users\fabio\TCC\11_peso_FINAL.csv
Agora você pode carregar este arquivo no seu notebook de junção.
